# Chapter 45: Storing Documents in ChromaDB — Documents, IDs and Embeddings

In Chapter 44, you learned ChromaDB fundamentals and collections. In this chapter, we'll focus specifically on how to store documents, unique IDs, and embeddings together so that each vector remains connected to its original text.
This notebook is completely independent: it creates its own sample dataset, loads its own embedding model, and initializes its own in-memory ChromaDB database. It does not import or read any previous chapter's files.

# Step 1 - Install the libraries 

In [ ]:
%pip install chromadb sentence-transformers numpy

chromadb stores the records, sentence-transformers generates embeddings, and numpy helps us inspect the vectors.

# Step 2 — Import the libraries

In [30]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

In [31]:
documents = [
    "Banks monitor cross-border payments for suspicious activity.",
    "Money laundering involves concealing the illegal origin of funds.",
    "Customer due diligence helps banks verify customer identities.",
    "Credit risk models estimate the probability of borrower default.",
    "Machine learning identifies unusual financial transaction patterns."
]

ids = [
    "banking_001",
    "banking_002",
    "banking_003",
    "banking_004",
    "banking_005"
]

metadatas = [
    {"category": "AML", "page": 1},
    {"category": "AML", "page": 2},
    {"category": "KYC", "page": 3},
    {"category": "Risk", "page": 4},
    {"category": "AI", "page": 5}
]
""

''

In [32]:
len(documents)

5

In [33]:
len(ids)

5

In [34]:
len(metadatas)

5

In [35]:
assert len(documents) == len(ids) == len(metadatas)
print("All records have matching lengths and unique IDs.")

All records have matching lengths and unique IDs.


In [36]:
assert len(set(ids)) == len(ids)

# This code is an assert statement used to verify that every single ID in your dataset is completely unique.

# Step 4 — Generate embeddings

In [37]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [38]:
embeddings = model.encode(documents)

In [39]:
embeddings.shape

(5, 384)

In [40]:
for i in range(len(documents)):
    print(f"\nID: {ids[i]}")
    print(f"Document: {documents[i]}")
    print(f"Vector dimensions: {len(embeddings[i])}")


ID: banking_001
Document: Banks monitor cross-border payments for suspicious activity.
Vector dimensions: 384

ID: banking_002
Document: Money laundering involves concealing the illegal origin of funds.
Vector dimensions: 384

ID: banking_003
Document: Customer due diligence helps banks verify customer identities.
Vector dimensions: 384

ID: banking_004
Document: Credit risk models estimate the probability of borrower default.
Vector dimensions: 384

ID: banking_005
Document: Machine learning identifies unusual financial transaction patterns.
Vector dimensions: 384


# Step 5 - Create an independent ChromaDB database

In [43]:
# EphemeralClient() creates an in-memory database, which is ideal for this exercise because it does not rely on any external database files.
# Rapid prototyping and testing. No files are written to the local disk, removing the need for cache cleanup.
client = chromadb.EphemeralClient()


In [44]:
client = chromadb.EphemeralClient()

# Delete if it exists to wipe out old data
try:
    client.delete_collection(name="chapter_45_document_storage")
except Exception:
    pass # Collection didn't exist yet, which is fine

collection = client.create_collection(
    name="chapter_45_document_storage",
    metadata={"hnsw:space": "cosine"}
)

print("Fresh collection created:", collection.name)

Fresh collection created: chapter_45_document_storage


# Step 6 — Store documents, IDs, metadata and embeddingsm

In [46]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings.tolist()
)

In [48]:
print(collection)

Collection(name=chapter_45_document_storage)


# Step 7 — Retrieve and verify the stored records

In [76]:
stored = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

In [77]:
stored

{'ids': ['banking_001',
  'banking_002',
  'banking_003',
  'banking_004',
  'banking_005'],
 'embeddings': array([[ 0.04494545, -0.04690323, -0.07358287, ..., -0.0453891 ,
          0.05228531, -0.04496431],
        [-0.02617313,  0.01649092, -0.04233609, ..., -0.01911972,
          0.15504289, -0.02148405],
        [-0.07805295,  0.00940214,  0.02780898, ...,  0.04361716,
          0.08624593, -0.13268182],
        [ 0.00924869, -0.0807237 , -0.02026054, ...,  0.02386039,
         -0.07186608, -0.04384258],
        [-0.03766765, -0.0257003 , -0.01018677, ..., -0.05917171,
          0.00782923, -0.01196884]]),
 'documents': ['Banks monitor cross-border payments for suspicious activity.',
  'Money laundering involves concealing the illegal origin of funds.',
  'Customer due diligence helps banks verify customer identities.',
  'Credit risk models estimate the probability of borrower default.',
  'Machine learning identifies unusual financial transaction patterns.'],
 'uris': None,
 'in

In [54]:
stored1 = collection.get(include=["documents"])

documents_only = stored1["documents"]


In [56]:
stored1


{'ids': ['banking_001',
  'banking_002',
  'banking_003',
  'banking_004',
  'banking_005'],
 'embeddings': None,
 'documents': ['Banks monitor cross-border payments for suspicious activity.',
  'Money laundering involves concealing the illegal origin of funds.',
  'Customer due diligence helps banks verify customer identities.',
  'Credit risk models estimate the probability of borrower default.',
  'Machine learning identifies unusual financial transaction patterns.'],
 'uris': None,
 'included': ['documents'],
 'data': None,
 'metadatas': None}

In [55]:
documents_only

['Banks monitor cross-border payments for suspicious activity.',
 'Money laundering involves concealing the illegal origin of funds.',
 'Customer due diligence helps banks verify customer identities.',
 'Credit risk models estimate the probability of borrower default.',
 'Machine learning identifies unusual financial transaction patterns.']

In [66]:
record = collection.get(
    ids=["banking_001"],
    include=["documents", "metadatas"]
)

In [67]:
record

{'ids': ['banking_001'],
 'embeddings': None,
 'documents': ['Banks monitor cross-border payments for suspicious activity.'],
 'uris': None,
 'included': ['documents', 'metadatas'],
 'data': None,
 'metadatas': [{'category': 'AML', 'page': 1}]}

# Step 8 — Verify document-to-embedding alignment

In [78]:
for record_id, document, vector in zip(
    stored["ids"],
    stored["documents"],
    stored["embeddings"]
):
    assert len(vector) == 384

    print(
        record_id,
        "|",
        document[:45],
        "| Dimensions:",
        len(vector)
    )

banking_001 | Banks monitor cross-border payments for suspi | Dimensions: 384
banking_002 | Money laundering involves concealing the ille | Dimensions: 384
banking_003 | Customer due diligence helps banks verify cus | Dimensions: 384
banking_004 | Credit risk models estimate the probability o | Dimensions: 384
banking_005 | Machine learning identifies unusual financial | Dimensions: 384


# THE END